In [6]:
import boto3

sagemaker_client = boto3.client("sagemaker", region_name="us-east-2")

# Delete endpoint config (this is what's blocking you)
try:
    sagemaker_client.delete_endpoint_config(EndpointConfigName="spotify-churn-endpoint-v3")
    sagemaker_client.delete_endpoint(EndpointName="spotify-churn-endpoint-v3")
    print("✓ Deleted endpoint config: spotify-churn-endpoint")
except Exception as e:
    print(f"Note: {e}")

# Also check for any other endpoints
try:
    response = sagemaker_client.list_endpoints()
    print("\nExisting endpoints:")
    for ep in response['Endpoints']:
        print(f"  - {ep['EndpointName']}: {ep['EndpointStatus']}")
        
    response2 = sagemaker_client.list_endpoint_configs()
    print("\nExisting endpoint configs:")
    for config in response2['EndpointConfigs']:
        print(f"  - {config['EndpointConfigName']}")
except Exception as e:
    print(f"Error listing: {e}")

Note: An error occurred (ValidationException) when calling the DeleteEndpoint operation: Cannot update in-progress endpoint "arn:aws:sagemaker:us-east-2:816595673043:endpoint/spotify-churn-endpoint-v6".

Existing endpoints:
  - spotify-churn-endpoint-v6: Creating
  - spotify-churn-endpoint-v3: Creating

Existing endpoint configs:


In [7]:
# Install dependencies
!pip install flaml scikit-learn pandas boto3 sagemaker joblib -q

import pandas as pd
import numpy as np
import boto3
import joblib
from flaml import AutoML
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import sagemaker
from sagemaker.sklearn import SKLearnModel

# ============================================================================
# CONFIGURATION
# ============================================================================
BUCKET = "mlopsfinalprojectdata"
TRAIN_KEY = "train.csv"
TEST_KEY = "test.csv"
REGION = "us-east-2"
ENDPOINT_NAME = "spotify-churn-endpoint-v7"  # Changed to avoid conflicts

s3 = boto3.client("s3", region_name=REGION)
sm_session = sagemaker.session.Session(boto_session=boto3.session.Session())
role_arn = sagemaker.get_execution_role()

print(f"✓ Role: {role_arn}")
print(f"✓ Region: {REGION}\n")

# ============================================================================
# 1. LOAD AND PREPROCESS TRAINING DATA
# ============================================================================
print("="*70)
print("STEP 1: Loading training data")
print("="*70)

obj = s3.get_object(Bucket=BUCKET, Key=TRAIN_KEY)
df_train = pd.read_csv(obj["Body"])

print(f"Shape: {df_train.shape}")
print(f"Target distribution:\n{df_train['is_churned'].value_counts()}\n")

# Get categorical columns
cat_cols = df_train.select_dtypes(include=["object"]).columns.tolist()
cat_cols = [c for c in cat_cols if c != "user_id"]

print(f"Categorical columns: {cat_cols}")

# One-hot encode
df_train_enc = pd.get_dummies(df_train, columns=cat_cols, drop_first=True)

# Prepare X and y
X_train = df_train_enc.drop(columns=["is_churned", "user_id"], errors="ignore")
y_train = df_train_enc["is_churned"]

print(f"✓ Features: {X_train.shape}")
print(f"✓ Feature names saved for later\n")

# ============================================================================
# 2. TRAIN MODEL WITH FLAML
# ============================================================================
print("="*70)
print("STEP 2: Training with FLAML (this takes ~5 minutes)")
print("="*70)

automl = AutoML()
automl.fit(
    X_train, 
    y_train,
    task="classification",
    time_budget=60,  # Just 1 minute for testing
    metric="f1",
    eval_method="cv",
    n_splits=3,  # Faster
    verbose=3,  # Show progress updates
    log_file_name="flaml_training.log"
)

model = automl.model

print(f"\n✓ Best model: {automl.best_estimator}")
print(f"✓ Best F1: {1 - automl.best_loss:.4f}")
print(f"✓ Best config: {automl.best_config}\n")

# ============================================================================
# 3. CREATE INFERENCE SCRIPT
# ============================================================================
print("="*70)
print("STEP 3: Creating inference script")
print("="*70)

inference_code = '''import joblib
import pandas as pd
import numpy as np
from io import StringIO
import os
import sys
import traceback
import subprocess

def model_fn(model_dir):
    """Load model with dependency installation."""
    import os, sys, subprocess
    print(f"Loading model from: {model_dir}")
    print(f"Files: {os.listdir(model_dir)}")
    
    # Install if missing (safe, idempotent)
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "flaml", "lightgbm", "xgboost"], 
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print("✓ Dependencies installed")
    except:
        print("Dependencies already exist or install skipped")
    
    import joblib
    model = joblib.load(os.path.join(model_dir, "model.pkl"))
    features = joblib.load(os.path.join(model_dir, "features.pkl"))
    print(f"✓ Model loaded: {type(model).__name__}, {len(features)} features")
    
    return {"model": model, "features": features}

def input_fn(request_body, content_type="text/csv"):
    """Parse CSV input."""
    try:
        if content_type == "text/csv":
            df = pd.read_csv(StringIO(request_body), header=None)
            print(f"Parsed input shape: {df.shape}")
            return df
        else:
            raise ValueError(f"Unsupported content type: {content_type}")
    except Exception as e:
        print(f"Error in input_fn: {str(e)}")
        raise

def predict_fn(input_data, model_dict):
    """Make predictions."""
    try:
        model = model_dict["model"]
        features = model_dict["features"]
        
        # Set column names
        input_data.columns = features
        
        print(f"Making predictions for {len(input_data)} samples")
        
        # Predict
        predictions = model.predict(input_data)
        probabilities = model.predict_proba(input_data)[:, 1]
        
        return pd.DataFrame({
            "prediction": predictions,
            "probability": probabilities
        })
    except Exception as e:
        print(f"Error in predict_fn: {str(e)}")
        print(traceback.format_exc())
        raise

def output_fn(prediction, accept="text/csv"):
    """Return CSV output."""
    try:
        return prediction.to_csv(index=False, header=False)
    except Exception as e:
        print(f"Error in output_fn: {str(e)}")
        raise
'''

with open("inference.py", "w") as f:
    f.write(inference_code)

print("✓ inference.py created\n")

# ============================================================================
# 4. PACKAGE AND UPLOAD MODEL
# ============================================================================
print("="*70)
print("STEP 4: Packaging model for SageMaker")
print("="*70)

# Save model and features
joblib.dump(model, "model.pkl")
joblib.dump(X_train.columns.tolist(), "features.pkl")
print("✓ Saved model.pkl and features.pkl")

# Verify files can be loaded
try:
    test_model = joblib.load("model.pkl")
    test_features = joblib.load("features.pkl")
    print(f"✓ Verification: Model type = {type(test_model)}")
    print(f"✓ Verification: {len(test_features)} features")
except Exception as e:
    print(f"❌ Error loading saved files: {e}")
    raise

# Create tarball with correct structure
import tarfile
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model.pkl", arcname="model.pkl")
    tar.add("features.pkl", arcname="features.pkl")

# Verify tarball contents
print("\nVerifying tarball contents:")
with tarfile.open("model.tar.gz", "r:gz") as tar:
    members = tar.getnames()
    print(f"  Files in tarball: {members}")
    if "model.pkl" not in members or "features.pkl" not in members:
        raise ValueError("Tarball missing required files!")
print("✓ Tarball verified")

# Upload to S3
model_s3_key = "model-artifacts/model.tar.gz"
s3.upload_file("model.tar.gz", BUCKET, model_s3_key)
model_s3_uri = f"s3://{BUCKET}/{model_s3_key}"

print(f"✓ Model uploaded to: {model_s3_uri}\n")

# ============================================================================
# 5. DEPLOY TO SAGEMAKER
# ============================================================================
print("="*70)
print("STEP 5: Deploying endpoint (takes 5-10 minutes)")
print("="*70)

# First, check if endpoint already exists and delete if failed
sagemaker_client = boto3.client("sagemaker", region_name=REGION)
try:
    existing = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    if existing['EndpointStatus'] == 'Failed':
        print(f"⚠ Found failed endpoint, deleting...")
        sagemaker_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
        sagemaker_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
        print("✓ Cleaned up failed endpoint\n")
    elif existing['EndpointStatus'] == 'InService':
        print(f"✓ Endpoint already exists and is InService!\n")
        # Skip deployment
        predictor = None
except:
    pass  # Endpoint doesn't exist, continue with deployment

# Test model locally before deploying
print("Testing model locally first...")
try:
    test_sample = X_train.head(5)
    test_pred = model.predict(test_sample)
    test_proba = model.predict_proba(test_sample)
    print(f"✓ Local test passed: {len(test_pred)} predictions made\n")
except Exception as e:
    print(f"❌ Local test failed: {e}")
    print("Model has issues - stopping deployment\n")
    raise

sklearn_model = SKLearnModel(
    model_data=model_s3_uri,
    role=role_arn,
    entry_point="inference.py",
    framework_version="1.2-1",
    py_version="py3",
    source_dir=".",
    sagemaker_session=sm_session
)

print("Starting deployment...")
try:
    predictor = sklearn_model.deploy(
        initial_instance_count=1,
        instance_type="ml.m5.large",
        endpoint_name=ENDPOINT_NAME,
        wait=True
    )
    print(f"✓ Endpoint '{ENDPOINT_NAME}' deployed!\n")
except Exception as e:
    print(f"\n❌ DEPLOYMENT FAILED: {e}\n")
    
    # Get detailed error info
    try:
        endpoint_desc = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
        print(f"Endpoint Status: {endpoint_desc['EndpointStatus']}")
        if 'FailureReason' in endpoint_desc:
            print(f"Failure Reason: {endpoint_desc['FailureReason']}")
    except:
        pass
    
    print("\nTroubleshooting tips:")
    print("1. Check CloudWatch logs for the endpoint")
    print("2. Verify model.tar.gz was uploaded correctly")
    print("3. Try a different instance type (ml.t2.medium)")
    raise

# ============================================================================
# 6. TEST ON ORIGINAL TEST SET
# ============================================================================
print("="*70)
print("STEP 6: Testing on original test set")
print("="*70)

# Load test data
obj_test = s3.get_object(Bucket=BUCKET, Key=TEST_KEY)
df_test_raw = pd.read_csv(obj_test["Body"])
y_true = df_test_raw["is_churned"].values

# Preprocess test data (same as training)
df_test_feats = df_test_raw.drop(columns=["is_churned", "user_id"], errors="ignore")
df_test_enc = pd.get_dummies(df_test_feats, columns=cat_cols, drop_first=True)

# Align columns with training
for col in X_train.columns:
    if col not in df_test_enc.columns:
        df_test_enc[col] = 0
X_test = df_test_enc[X_train.columns]

print(f"Test data shape: {X_test.shape}")

# Run inference
runtime = boto3.client("sagemaker-runtime", region_name=REGION)
payload = X_test.to_csv(index=False, header=False).encode("utf-8")

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="text/csv",
    Body=payload
)

# Parse results
result_csv = response["Body"].read().decode("utf-8")
lines = result_csv.strip().split("\n")
predictions = [int(line.split(",")[0]) for line in lines]
probabilities = [float(line.split(",")[1]) for line in lines]

# Calculate metrics
acc = accuracy_score(y_true, predictions)
f1 = f1_score(y_true, predictions)
auc = roc_auc_score(y_true, probabilities)

print("\n📊 ORIGINAL TEST RESULTS:")
print(f"Accuracy:  {acc:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}\n")
print(classification_report(y_true, predictions, target_names=["Not Churned", "Churned"]))

# ============================================================================
# 7. TEST WITH MODIFIED FEATURES
# ============================================================================
print("="*70)
print("STEP 7: Testing with modified features")
print("="*70)

df_modified = df_test_raw.copy()
df_modified["listening_time"] = df_modified["listening_time"] * 1.5
df_modified["skip_rate"] = (df_modified["skip_rate"] + 0.05).clip(0, 1)

print("Changes:")
print("  - listening_time: +50%")
print("  - skip_rate: +0.05\n")

# Preprocess
df_mod_feats = df_modified.drop(columns=["is_churned", "user_id"], errors="ignore")
df_mod_enc = pd.get_dummies(df_mod_feats, columns=cat_cols, drop_first=True)

for col in X_train.columns:
    if col not in df_mod_enc.columns:
        df_mod_enc[col] = 0
X_modified = df_mod_enc[X_train.columns]

# Inference
payload_mod = X_modified.to_csv(index=False, header=False).encode("utf-8")
response_mod = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="text/csv",
    Body=payload_mod
)

result_mod_csv = response_mod["Body"].read().decode("utf-8")
lines_mod = result_mod_csv.strip().split("\n")
predictions_mod = [int(line.split(",")[0]) for line in lines_mod]
probabilities_mod = [float(line.split(",")[1]) for line in lines_mod]

# Metrics
acc_mod = accuracy_score(y_true, predictions_mod)
f1_mod = f1_score(y_true, predictions_mod)
auc_mod = roc_auc_score(y_true, probabilities_mod)

print("📊 MODIFIED TEST RESULTS:")
print(f"Accuracy:  {acc_mod:.4f}")
print(f"F1 Score:  {f1_mod:.4f}")
print(f"ROC AUC:   {auc_mod:.4f}\n")

changes = sum(p1 != p2 for p1, p2 in zip(predictions, predictions_mod))
print(f"Predictions changed: {changes}/{len(predictions)} ({changes/len(predictions)*100:.1f}%)\n")

print(classification_report(y_true, predictions_mod, target_names=["Not Churned", "Churned"]))

# ============================================================================
# 8. CLEANUP - DELETE ENDPOINT
# ============================================================================
print("="*70)
print("STEP 8: CLEANUP - Deleting endpoint to save costs")
print("="*70)

sagemaker_client = boto3.client("sagemaker", region_name=REGION)

try:
    sagemaker_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print(f"✓ Deleted endpoint: {ENDPOINT_NAME}")
    
    sagemaker_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print(f"✓ Deleted endpoint config: {ENDPOINT_NAME}")
    
    print("\n💰 Resources cleaned up - no more charges!")
    
except Exception as e:
    print(f"⚠ Cleanup note: {e}")
    print("\nManual cleanup if needed:")
    print(f"  aws sagemaker delete-endpoint --endpoint-name {ENDPOINT_NAME}")
    print(f"  aws sagemaker delete-endpoint-config --endpoint-config-name {ENDPOINT_NAME}")

print("\n" + "="*70)
print("✅ COMPLETE!")
print("="*70)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
✓ Role: arn:aws:iam::816595673043:role/service-role/AmazonSageMakerAdminIAMExecutionRole
✓ Region: us-east-2

STEP 1: Loading training data
Shape: (6400, 12)
Target distribution:
is_churned
0    4743
1    1657
Name: count, dtype: int64

Categorical columns: ['gender', 'country', 'subscription_type', 'device_type']
✓ Features: (6400, 20)
✓ Feature names saved for later

STEP 2: Training with FLAML (this takes ~5 minutes)
[flaml.automl.logger: 11-27 19:17:39] {1752} INFO - task = classification
[flaml.automl.logger: 11-27 19:17:39] {1763} INFO - 